# Buscador de Archivos

Proyecto en **Python puro** (`pathlib` + `re`).

Busca archivos por:
- Nombre (soporta expresiones regulares)
- Extensión
- Contenido de texto (regex)

Ejecuta las celdas en orden.

In [ ]:
import re
from pathlib import Path

def buscar_archivos(
    carpeta: str,
    nombre: str = None,
    extension: str = None,
    contenido: str = None,
    ignorar_mayus: bool = True,
    recursivo: bool = True,
) -> list:
    """Busca archivos que coincidan con los criterios."""
    carpeta_path = Path(carpeta)
    if not carpeta_path.exists() or not carpeta_path.is_dir():
        raise ValueError(f"Carpeta no válida: {carpeta}")

    flags = re.IGNORECASE if ignorar_mayus else 0
    patron_nombre = re.compile(nombre, flags) if nombre else None
    patron_contenido = re.compile(contenido, flags) if contenido else None

    if extension and not extension.startswith("."):
        extension = "." + extension

    resultados = []
    patron_busqueda = "**/*" if recursivo else "*"

    for archivo in carpeta_path.glob(patron_busqueda):
        if not archivo.is_file():
            continue

        if extension and archivo.suffix.lower() != extension.lower():
            continue

        if patron_nombre and not patron_nombre.search(archivo.name):
            continue

        if patron_contenido:
            try:
                texto = archivo.read_text(encoding="utf-8", errors="ignore")
                if not patron_contenido.search(texto):
                    continue
            except Exception:
                continue

        resultados.append(archivo)

    return resultados

def mostrar_resultados(resultados: list):
    if not resultados:
        print("No se encontraron archivos.")
        return

    print(f"\nSe encontraron {len(resultados)} archivo(s):\n")
    for i, archivo in enumerate(resultados, 1):
        print(f"{i:3}. {archivo}")

print("Funciones cargadas: buscar_archivos() y mostrar_resultados()")

## Ejemplos rápidos
En Colab la carpeta actual suele ser `/content`.

In [ ]:
# Buscar todos los .py en la carpeta actual (no recursivo)
resultados = buscar_archivos(".", extension=".py", recursivo=False)
mostrar_resultados(resultados)

# Buscar archivos cuyo nombre contenga "test"
# resultados = buscar_archivos("/content", nombre="test", recursivo=True)
# mostrar_resultados(resultados)

## Modo interactivo
Ejecuta la siguiente celda y responde las preguntas.

In [ ]:
print("=== Buscador de Archivos ===\n")
carpeta = input("Carpeta a buscar (Enter = actual): ").strip() or "."
nombre = input("Patrón de nombre (regex, opcional): ").strip() or None
extension = input("Extensión (ej: .py, opcional): ").strip() or None
contenido = input("Texto en contenido (regex, opcional): ").strip() or None
recursivo = input("Buscar en subcarpetas? (s/n, defecto s): ").lower() != "n"

resultados = buscar_archivos(carpeta, nombre, extension, contenido, True, recursivo)
mostrar_resultados(resultados)